# GPU Programming Tutorial — CUDA C++ (Kaggle)
**How to use:** 1) In this notebook click *Settings* (gear icon) → *Accelerator* → **GPU** → Save. 2) *Run* → *Run All*. 3) When finished, *File* → *Download as .ipynb* and send the downloaded file back — it contains all code **and** the real outputs as evidence.
If `nvcc -arch=native` fails, replace `-arch=native` with `-arch=sm_70` (or `sm_75` for T4) in the compile cells.


## 0. Environment check


In [103]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Shared header (create once)


In [104]:
%%writefile cuda_check.cuh
#pragma once
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>

inline void cuda_check(cudaError_t error, const char *file, int line)
{
    if (error != cudaSuccess) {
        std::fprintf(stderr, "%s:%d: CUDA error: %s\n",
                     file, line, cudaGetErrorString(error));
        std::exit(EXIT_FAILURE);
    }
}
#define CUDA_CHECK(call) cuda_check((call), __FILE__, __LINE__)

inline bool close_enough(double got, double expected,
                         double atol = 1e-5, double rtol = 1e-5)
{
    return std::isfinite(got) && std::isfinite(expected) &&
           std::fabs(got - expected) <= atol + rtol * std::fabs(expected);
}



Overwriting cuda_check.cuh


## Exercise 1 — Inspect the available GPU


In [105]:
%%writefile device_info.cu
#include "cuda_check.cuh"

int main()
{
    int count = 0;
    CUDA_CHECK(cudaGetDeviceCount(&count));
    if (count == 0) {
        std::fprintf(stderr, "No CUDA device is available.\n");
        return 1;
    }
    CUDA_CHECK(cudaSetDevice(0));
    cudaDeviceProp p{};
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    std::printf("Device: %s\n", p.name);
    std::printf("Compute capability: %d.%d\n", p.major, p.minor);
    std::printf("SM count: %d\n", p.multiProcessorCount);
    std::printf("Warp size: %d\n", p.warpSize);
    std::printf("Maximum threads per block: %d\n", p.maxThreadsPerBlock);
    std::printf("Global memory: %.2f GiB\n",
                p.totalGlobalMem / (1024.0 * 1024.0 * 1024.0));
    std::printf("Shared memory per block: %zu bytes\n",
                static_cast<std::size_t>(p.sharedMemPerBlock));
    return 0;
}



Overwriting device_info.cu


In [106]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native device_info.cu -o device_info
!./device_info


Device: Tesla T4
Compute capability: 7.5
SM count: 40
Warp size: 32
Maximum threads per block: 1024
Global memory: 14.56 GiB
Shared memory per block: 49152 bytes


## Exercise 2 — Launch a first kernel


In [107]:
%%writefile hello_gpu.cu
#include "cuda_check.cuh"

__global__ void hello(int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        printf("block=%u local=%u global=%d\n",
               blockIdx.x, threadIdx.x, i);
    }
}

int main()
{
    const int n = 20;
    const int threads = 8;
    const int blocks = (n + threads - 1) / threads;
    hello<<<blocks, threads>>>(n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    std::printf("Host: the kernel has finished.\n");
    return 0;
}



Overwriting hello_gpu.cu


In [108]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native hello_gpu.cu -o hello_gpu
!./hello_gpu


block=2 local=0 global=16
block=2 local=1 global=17
block=2 local=2 global=18
block=2 local=3 global=19
block=0 local=0 global=0
block=0 local=1 global=1
block=0 local=2 global=2
block=0 local=3 global=3
block=0 local=4 global=4
block=0 local=5 global=5
block=0 local=6 global=6
block=0 local=7 global=7
block=1 local=0 global=8
block=1 local=1 global=9
block=1 local=2 global=10
block=1 local=3 global=11
block=1 local=4 global=12
block=1 local=5 global=13
block=1 local=6 global=14
block=1 local=7 global=15
Host: the kernel has finished.


### Variant: n=17, threads=16 (predict: 2 blocks, 32 threads launched, 17 active)


In [109]:
%%writefile hello_gpu_17.cu
#include "cuda_check.cuh"

__global__ void hello(int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        printf("block=%u local=%u global=%d\n",
               blockIdx.x, threadIdx.x, i);
    }
}

int main()
{
    const int n = 17;
    const int threads = 16;
    const int blocks = (n + threads - 1) / threads;
    std::printf("n=%d threads=%d blocks=%d launched=%d\n",
                n, threads, blocks, blocks * threads);
    hello<<<blocks, threads>>>(n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    std::printf("Host: the kernel has finished.\n");
    return 0;
}



Overwriting hello_gpu_17.cu


In [110]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native hello_gpu_17.cu -o hello_gpu_17
!./hello_gpu_17


n=17 threads=16 blocks=2 launched=32
block=1 local=0 global=16
block=0 local=0 global=0
block=0 local=1 global=1
block=0 local=2 global=2
block=0 local=3 global=3
block=0 local=4 global=4
block=0 local=5 global=5
block=0 local=6 global=6
block=0 local=7 global=7
block=0 local=8 global=8
block=0 local=9 global=9
block=0 local=10 global=10
block=0 local=11 global=11
block=0 local=12 global=12
block=0 local=13 global=13
block=0 local=14 global=14
block=0 local=15 global=15
Host: the kernel has finished.


## Exercise 3 — Vector add with explicit device memory


In [111]:
%%writefile vector_add.cu
#include "cuda_check.cuh"
#include <vector>

#ifndef N_VAL
#define N_VAL 1000003
#endif

__global__ void vector_add(const float *a, const float *b,
                           float *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main()
{
    const int n = N_VAL;
    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    const std::size_t bytes = static_cast<std::size_t>(n) * sizeof(float);
    std::vector<float> a(n), b(n), c(n);
    for (int i = 0; i < n; ++i) {
        a[i] = (i % 97) * 0.25f;
        b[i] = (i % 31) * 0.5f;
    }
    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), bytes));

    // GPU WORK: replace this block in Exercise 4.
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, bytes, cudaMemcpyDeviceToHost));
    // END GPU WORK

    int errors = 0;
    for (int i = 0; i < n; ++i) {
        double expected = static_cast<double>(a[i]) + b[i];
        if (!close_enough(c[i], expected)) ++errors;
    }
    std::printf("n=%d blocks=%d threads=%d errors=%d\n",
                n, blocks, threads, errors);
    std::printf("first=%.2f last=%.2f\n", c.front(), c.back());
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting vector_add.cu


In [112]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native vector_add.cu -o vector_add
!./vector_add


n=1000003 blocks=3907 threads=256 errors=0
first=0.00 last=9.25


### Edge sizes (recompile per size, as the tutorial asks)


In [113]:
%%bash
for n in 1 255 256 257 1000003; do nvcc -std=c++17 -O3 -lineinfo -arch=native -DN_VAL=$n vector_add.cu -o vector_add && ./vector_add; done


n=1 blocks=1 threads=256 errors=0
first=0.00 last=0.00
n=255 blocks=1 threads=256 errors=0
first=0.00 last=18.00
n=256 blocks=1 threads=256 errors=0
first=0.00 last=18.75
n=257 blocks=2 threads=256 errors=0
first=0.00 last=19.50
n=1000003 blocks=3907 threads=256 errors=0
first=0.00 last=9.25


### Modification: c[i] = 2*a[i] + b[i]


In [114]:
%%writefile vector_add_alpha.cu
#include "cuda_check.cuh"
#include <vector>

// Exercise 3 modification: c[i] = 2*a[i] + b[i]
__global__ void vector_add(const float *a, const float *b,
                           float *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = 2.0f * a[i] + b[i];
}

int main()
{
    const int n = 1000003;
    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    const std::size_t bytes = static_cast<std::size_t>(n) * sizeof(float);
    std::vector<float> a(n), b(n), c(n);
    for (int i = 0; i < n; ++i) {
        a[i] = (i % 97) * 0.25f;
        b[i] = (i % 31) * 0.5f;
    }
    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, bytes, cudaMemcpyDeviceToHost));

    int errors = 0;
    for (int i = 0; i < n; ++i) {
        double expected = 2.0 * static_cast<double>(a[i]) + b[i];
        if (!close_enough(c[i], expected)) ++errors;
    }
    std::printf("alpha=2: n=%d blocks=%d threads=%d errors=%d\n",
                n, blocks, threads, errors);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting vector_add_alpha.cu


In [115]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native vector_add_alpha.cu -o vector_add_alpha
!./vector_add_alpha


alpha=2: n=1000003 blocks=3907 threads=256 errors=0


## Exercise 4 — Kernel time vs transfer-inclusive time
5 runs per size; medians computed below. `kernel` = GPU events around the kernel only; `operation` = H2D + kernel + sync + D2H (wall clock).


In [116]:
%%writefile vector_benchmark.cu
#include "cuda_check.cuh"
#include <vector>
#include <chrono>

#ifndef N_VAL
#define N_VAL 1000003
#endif

__global__ void vector_add(const float *a, const float *b,
                           float *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main()
{
    const int n = N_VAL;
    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    const std::size_t bytes = static_cast<std::size_t>(n) * sizeof(float);
    std::vector<float> a(n), b(n), c(n);
    for (int i = 0; i < n; ++i) {
        a[i] = (i % 97) * 0.25f;
        b[i] = (i % 31) * 0.5f;
    }

    // Serial reference and timer (Edit 2).
    using Clock = std::chrono::steady_clock;
    std::vector<float> reference(n);
    auto cpu_start = Clock::now();
    for (int i = 0; i < n; ++i) {
        reference[i] = a[i] + b[i];
    }
    double cpu_ms = std::chrono::duration<double, std::milli>(
        Clock::now() - cpu_start).count();

    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), bytes));

    // Replacement GPU WORK block (Edit 3).
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    // Warm up runtime and kernel before measuring.
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    auto wall_start = Clock::now();
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(start));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, bytes, cudaMemcpyDeviceToHost));
    double operation_ms = std::chrono::duration<double, std::milli>(
        Clock::now() - wall_start).count();

    float kernel_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&kernel_ms, start, stop));
    std::printf("n=%d CPU=%.6f ms kernel=%.6f ms operation=%.6f ms\n",
                n, cpu_ms, kernel_ms, operation_ms);
    if (kernel_ms > 0.0f && operation_ms > 0.0) {
        std::printf("resident-data speedup=%.3f operation speedup=%.3f\n",
                    cpu_ms / kernel_ms, cpu_ms / operation_ms);
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    int errors = 0;
    for (int i = 0; i < n; ++i) {
        double expected = reference[i];
        if (!close_enough(c[i], expected)) ++errors;
    }
    std::printf("errors=%d\n", errors);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting vector_benchmark.cu


In [117]:
%%bash
for n in 1003 100003 1000003 10000003; do for rep in 1 2 3 4 5; do nvcc -std=c++17 -O3 -lineinfo -arch=native -DN_VAL=$n vector_benchmark.cu -o vector_benchmark && ./vector_benchmark; done; done 2>&1 | tee ex4_raw.txt


n=1003 CPU=0.000395 ms kernel=0.008192 ms operation=0.047034 ms
resident-data speedup=0.048 operation speedup=0.008
errors=0
n=1003 CPU=0.000347 ms kernel=0.008032 ms operation=0.044569 ms
resident-data speedup=0.043 operation speedup=0.008
errors=0
n=1003 CPU=0.000393 ms kernel=0.008192 ms operation=0.043703 ms
resident-data speedup=0.048 operation speedup=0.009
errors=0
n=1003 CPU=0.000385 ms kernel=0.007872 ms operation=0.044675 ms
resident-data speedup=0.049 operation speedup=0.009
errors=0
n=1003 CPU=0.000467 ms kernel=0.008128 ms operation=0.043836 ms
resident-data speedup=0.057 operation speedup=0.011
errors=0
n=100003 CPU=0.038521 ms kernel=0.015520 ms operation=0.397477 ms
resident-data speedup=2.482 operation speedup=0.097
errors=0
n=100003 CPU=0.031668 ms kernel=0.013824 ms operation=0.422650 ms
resident-data speedup=2.291 operation speedup=0.075
errors=0
n=100003 CPU=0.040608 ms kernel=0.010976 ms operation=0.407043 ms
resident-data speedup=3.700 operation speedup=0.100
err

In [118]:
import re, statistics
runs = {}
cur = None
for line in open('ex4_raw.txt'):
    m = re.search(r'n=(\d+) CPU=([\d.]+) ms kernel=([\d.]+) ms operation=([\d.]+) ms', line)
    if m:
        n = int(m.group(1)); runs.setdefault(n, []).append(tuple(map(float, m.groups()[1:])))
print(f"{'n':>10} {'CPU(ms)':>10} {'kernel(ms)':>12} {'oper(ms)':>10} {'Sres':>7} {'Sop':>7}")
for n in sorted(runs):
    cpu = statistics.median(r[0] for r in runs[n])
    ker = statistics.median(r[1] for r in runs[n])
    opr = statistics.median(r[2] for r in runs[n])
    print(f"{n:>10} {cpu:>10.4f} {ker:>12.4f} {opr:>10.4f} {cpu/ker:>7.2f} {cpu/opr:>7.2f}")


         n    CPU(ms)   kernel(ms)   oper(ms)    Sres     Sop
      1003     0.0004       0.0081     0.0446    0.05    0.01
    100003     0.0385       0.0144     0.4070    2.68    0.09
   1000003     0.5826       0.0489     3.0164   11.92    0.19
  10000003     8.7234       0.4574    27.5649   19.07    0.32


## Exercise 5 — Grid-stride kernel, launch tuning, effective bandwidth
Same timed host program as Exercise 4, only the kernel is grid-stride. Effective bandwidth = 12n / Tk (GB/s).


In [119]:
%%writefile vector_gridstride.cu
#include "cuda_check.cuh"
#include <vector>
#include <chrono>

// Exercise 5: grid-stride kernel. Same host program as vector_benchmark.cu
// (timed) so kernel times and effective bandwidth can be compared across
// launch configurations.
#ifndef N_VAL
#define N_VAL 1000003
#endif
#ifndef BLOCKS_VAL
#define BLOCKS_VAL 128
#endif
#ifndef THREADS_VAL
#define THREADS_VAL 256
#endif

__global__ void vector_add(const float *a, const float *b,
                           float *c, int n)
{
    int first = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = first; i < n; i += stride) {
        c[i] = a[i] + b[i];
    }
}

int main()
{
    const int n = N_VAL;
    const int threads = THREADS_VAL;
    const int blocks = BLOCKS_VAL;
    const std::size_t bytes = static_cast<std::size_t>(n) * sizeof(float);
    std::vector<float> a(n), b(n), c(n);
    for (int i = 0; i < n; ++i) {
        a[i] = (i % 97) * 0.25f;
        b[i] = (i % 31) * 0.5f;
    }

    using Clock = std::chrono::steady_clock;
    std::vector<float> reference(n);
    auto cpu_start = Clock::now();
    for (int i = 0; i < n; ++i) {
        reference[i] = a[i] + b[i];
    }
    double cpu_ms = std::chrono::duration<double, std::milli>(
        Clock::now() - cpu_start).count();

    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), bytes));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    // Warm up runtime and kernel before measuring.
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    auto wall_start = Clock::now();
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(start));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, bytes, cudaMemcpyDeviceToHost));
    double operation_ms = std::chrono::duration<double, std::milli>(
        Clock::now() - wall_start).count();

    float kernel_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&kernel_ms, start, stop));
    double gbps = (12.0 * n) / (kernel_ms * 1e6); // 12n bytes / Tk
    std::printf("n=%d blocks=%d threads=%d CPU=%.6f ms kernel=%.6f ms "
                "operation=%.6f ms eff_bw=%.2f GB/s\n",
                n, blocks, threads, cpu_ms, kernel_ms, operation_ms, gbps);
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    int errors = 0;
    for (int i = 0; i < n; ++i) {
        double expected = reference[i];
        if (!close_enough(c[i], expected)) ++errors;
    }
    std::printf("errors=%d\n", errors);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting vector_gridstride.cu


In [120]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native vector_gridstride.cu -o vector_gridstride
!./vector_gridstride


n=1000003 blocks=128 threads=256 CPU=0.555602 ms kernel=0.055776 ms operation=3.029862 ms eff_bw=215.15 GB/s
errors=0


### Sweep: blocks × threads (all within device limits)


In [121]:
%%bash
for b in 64 128 256; do for t in 64 128 256; do for rep in 1 2 3; do nvcc -std=c++17 -O3 -lineinfo -arch=native -DBLOCKS_VAL=$b -DTHREADS_VAL=$t vector_gridstride.cu -o vector_gridstride && ./vector_gridstride; done; done; done 2>&1 | tee ex5_raw.txt


n=1000003 blocks=64 threads=64 CPU=0.603741 ms kernel=0.166144 ms operation=3.128801 ms eff_bw=72.23 GB/s
errors=0
n=1000003 blocks=64 threads=64 CPU=0.489420 ms kernel=0.161856 ms operation=3.075083 ms eff_bw=74.14 GB/s
errors=0
n=1000003 blocks=64 threads=64 CPU=0.529759 ms kernel=0.169472 ms operation=3.048073 ms eff_bw=70.81 GB/s
errors=0
n=1000003 blocks=64 threads=128 CPU=0.557555 ms kernel=0.107008 ms operation=3.092441 ms eff_bw=112.14 GB/s
errors=0
n=1000003 blocks=64 threads=128 CPU=0.560553 ms kernel=0.102912 ms operation=2.858953 ms eff_bw=116.60 GB/s
errors=0
n=1000003 blocks=64 threads=128 CPU=0.496525 ms kernel=0.101024 ms operation=3.019443 ms eff_bw=118.78 GB/s
errors=0
n=1000003 blocks=64 threads=256 CPU=0.588154 ms kernel=0.063200 ms operation=3.059508 ms eff_bw=189.87 GB/s
errors=0
n=1000003 blocks=64 threads=256 CPU=0.691374 ms kernel=0.061440 ms operation=3.133988 ms eff_bw=195.31 GB/s
errors=0
n=1000003 blocks=64 threads=256 CPU=0.507318 ms kernel=0.059040 ms ope

In [122]:
import re, statistics
runs = {}
for line in open('ex5_raw.txt'):
    m = re.search(r'n=\d+ blocks=(\d+) threads=(\d+) .* kernel=([\d.]+) ms .* eff_bw=([\d.]+) GB/s', line)
    if m:
        runs.setdefault((m.group(1), m.group(2)), []).append((float(m.group(3)), float(m.group(4))))
print(f"{'blocks':>7} {'threads':>8} {'kernel(ms)':>11} {'GB/s':>8}")
for (b, t), v in sorted(runs.items(), key=lambda kv: statistics.median(x[0] for x in kv[1])):
    print(f"{b:>7} {t:>8} {statistics.median(x[0] for x in v):>11.4f} {statistics.median(x[1] for x in v):>8.2f}")


 blocks  threads  kernel(ms)     GB/s
    256      128      0.0516   232.63
    128      256      0.0519   231.34
    256      256      0.0550   218.15
     64      256      0.0614   195.31
    128      128      0.0636   188.82
    256       64      0.0657   182.66
    128       64      0.1018   117.92
     64      128      0.1029   116.60
     64       64      0.1661    72.23


## Exercise 6 — Reduction with shared memory


In [123]:
%%writefile reduce_sum.cu
#include "cuda_check.cuh"
#include <vector>

#ifndef N_VAL
#define N_VAL 1000003
#endif

constexpr int BLOCK = 256;
// Must be a power of two for this algorithm.

__global__ void block_sum(const int *a, int *partial, int n)
{
    __shared__ int values[BLOCK];
    int t = threadIdx.x;
    int i = blockIdx.x * blockDim.x + t;
    values[t] = (i < n) ? a[i] : 0;
    __syncthreads();

    for (int offset = BLOCK / 2; offset > 0; offset /= 2) {
        if (t < offset) values[t] += values[t + offset];
        __syncthreads();
    }
    if (t == 0) partial[blockIdx.x] = values[0];
}

int main()
{
    const int n = N_VAL;
    const int blocks = (n + BLOCK - 1) / BLOCK;
    std::vector<int> a(n), partial(blocks);
    long long expected = 0;
    for (int i = 0; i < n; ++i) {
        a[i] = i % 7;
        expected += a[i];
    }
    int *d_a = nullptr, *d_partial = nullptr;
    std::size_t bytes = static_cast<std::size_t>(n) * sizeof(int);
    std::size_t partial_bytes =
        static_cast<std::size_t>(blocks) * sizeof(int);
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_partial),
                          partial_bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    block_sum<<<blocks, BLOCK>>>(d_a, d_partial, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(partial.data(), d_partial, partial_bytes,
                          cudaMemcpyDeviceToHost));
    long long result = 0;
    for (int x : partial) result += x;
    std::printf("n=%d expected=%lld result=%lld match=%s\n",
                n, expected, result, expected == result ? "yes" : "no");
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_partial));
    return result != expected;
}



Overwriting reduce_sum.cu


In [124]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native reduce_sum.cu -o reduce_sum
!./reduce_sum


n=1000003 expected=3000003 result=3000003 match=yes


### Edge sizes


In [125]:
%%bash
for n in 1 255 256 257 1000003; do nvcc -std=c++17 -O3 -lineinfo -arch=native -DN_VAL=$n reduce_sum.cu -o reduce_sum && ./reduce_sum; done


n=1 expected=0 result=0 match=yes
n=255 expected=759 result=759 match=yes
n=256 expected=762 result=762 match=yes
n=257 expected=766 result=766 match=yes
n=1000003 expected=3000003 result=3000003 match=yes


## Exercise 7 — Matrix multiplication: naive vs tiled


In [126]:
%%writefile matrix_mul.cu
#include "cuda_check.cuh"
#include <vector>

__global__ void matmul(const float *a, const float *b, float *c,
                       int m, int k, int n)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < m && col < n) {
        float sum = 0.0f;
        for (int q = 0; q < k; ++q) {
            sum += a[row * k + q] * b[q * n + col];
        }
        c[row * n + col] = sum;
    }
}

int main()
{
    const int m = 127, k = 130, n = 129;
    std::vector<float> a(m * k), b(k * n), c(m * n);
    for (int i = 0; i < m * k; ++i) a[i] = ((i % 17) - 8) * 0.125f;
    for (int i = 0; i < k * n; ++i) b[i] = ((i % 13) - 6) * 0.0625f;
    std::size_t a_bytes = a.size() * sizeof(float);
    std::size_t b_bytes = b.size() * sizeof(float);
    std::size_t c_bytes = c.size() * sizeof(float);
    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), a_bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), b_bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), c_bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), a_bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), b_bytes, cudaMemcpyHostToDevice));
    dim3 block(16, 16);
    dim3 grid((n + block.x - 1) / block.x,
              (m + block.y - 1) / block.y);
    matmul<<<grid, block>>>(d_a, d_b, d_c, m, k, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, c_bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    double max_error = 0.0;
    for (int row = 0; row < m; ++row) {
        for (int col = 0; col < n; ++col) {
            double reference = 0.0;
            for (int q = 0; q < k; ++q) {
                reference += static_cast<double>(a[row * k + q]) *
                             b[q * n + col];
            }
            double got = c[row * n + col];
            double error = std::fabs(got - reference);
            if (error > max_error) max_error = error;
            if (!close_enough(got, reference, 1e-4, 1e-4)) ++errors;
        }
    }
    std::printf("M=%d K=%d N=%d errors=%d max_error=%.3e\n",
                m, k, n, errors, max_error);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting matrix_mul.cu


In [127]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native matrix_mul.cu -o matrix_mul
!./matrix_mul


M=127 K=130 N=129 errors=0 max_error=0.000e+00


In [128]:
%%writefile matrix_mul_tiled.cu
#include "cuda_check.cuh"
#include <vector>

// Tiled variant of matrix_mul.cu: same host code, replacement kernel.
constexpr int TILE = 16;

__global__ void matmul(const float *a, const float *b, float *c,
                       int m, int k, int n)
{
    __shared__ float a_tile[TILE][TILE];
    __shared__ float b_tile[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int col = blockIdx.x * TILE + tx;
    int row = blockIdx.y * TILE + ty;
    float sum = 0.0f;

    for (int base = 0; base < k; base += TILE) {
        int a_col = base + tx;
        int b_row = base + ty;
        a_tile[ty][tx] = (row < m && a_col < k)
            ? a[row * k + a_col] : 0.0f;
        b_tile[ty][tx] = (b_row < k && col < n)
            ? b[b_row * n + col] : 0.0f;
        __syncthreads();
        for (int q = 0; q < TILE; ++q) {
            sum += a_tile[ty][q] * b_tile[q][tx];
        }
        __syncthreads();
    }
    if (row < m && col < n) c[row * n + col] = sum;
}

int main()
{
    const int m = 127, k = 130, n = 129;
    std::vector<float> a(m * k), b(k * n), c(m * n);
    for (int i = 0; i < m * k; ++i) a[i] = ((i % 17) - 8) * 0.125f;
    for (int i = 0; i < k * n; ++i) b[i] = ((i % 13) - 6) * 0.0625f;
    std::size_t a_bytes = a.size() * sizeof(float);
    std::size_t b_bytes = b.size() * sizeof(float);
    std::size_t c_bytes = c.size() * sizeof(float);
    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), a_bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), b_bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), c_bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), a_bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), b_bytes, cudaMemcpyHostToDevice));
    dim3 block(16, 16);
    dim3 grid((n + block.x - 1) / block.x,
              (m + block.y - 1) / block.y);
    matmul<<<grid, block>>>(d_a, d_b, d_c, m, k, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, c_bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    double max_error = 0.0;
    for (int row = 0; row < m; ++row) {
        for (int col = 0; col < n; ++col) {
            double reference = 0.0;
            for (int q = 0; q < k; ++q) {
                reference += static_cast<double>(a[row * k + q]) *
                             b[q * n + col];
            }
            double got = c[row * n + col];
            double error = std::fabs(got - reference);
            if (error > max_error) max_error = error;
            if (!close_enough(got, reference, 1e-4, 1e-4)) ++errors;
        }
    }
    std::printf("TILED M=%d K=%d N=%d errors=%d max_error=%.3e\n",
                m, k, n, errors, max_error);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting matrix_mul_tiled.cu


In [129]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native matrix_mul_tiled.cu -o matrix_mul_tiled
!./matrix_mul_tiled


TILED M=127 K=130 N=129 errors=0 max_error=0.000e+00


### Timed comparison on square sizes (warm-up outside, CUDA events, 5 runs)


In [130]:
%%writefile matrix_bench.cu
#include "cuda_check.cuh"
#include <vector>
#include <chrono>
#include <string>

// Timed square matrix multiplication for Exercise 7 tasks 2-3.
// Select kernel with -DUSE_TILED (default: naive). Size via -DN_VAL.
// Warm-up outside the timed interval; CUDA events around the kernel.
#ifndef N_VAL
#define N_VAL 256
#endif
constexpr int TILE = 16;

__global__ void matmul_naive(const float *a, const float *b, float *c,
                             int m, int k, int n)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < m && col < n) {
        float sum = 0.0f;
        for (int q = 0; q < k; ++q) {
            sum += a[row * k + q] * b[q * n + col];
        }
        c[row * n + col] = sum;
    }
}

__global__ void matmul_tiled(const float *a, const float *b, float *c,
                             int m, int k, int n)
{
    __shared__ float a_tile[TILE][TILE];
    __shared__ float b_tile[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int col = blockIdx.x * TILE + tx;
    int row = blockIdx.y * TILE + ty;
    float sum = 0.0f;
    for (int base = 0; base < k; base += TILE) {
        int a_col = base + tx;
        int b_row = base + ty;
        a_tile[ty][tx] = (row < m && a_col < k)
            ? a[row * k + a_col] : 0.0f;
        b_tile[ty][tx] = (b_row < k && col < n)
            ? b[b_row * n + col] : 0.0f;
        __syncthreads();
        for (int q = 0; q < TILE; ++q) {
            sum += a_tile[ty][q] * b_tile[q][tx];
        }
        __syncthreads();
    }
    if (row < m && col < n) c[row * n + col] = sum;
}

int main()
{
    const int m = N_VAL, k = N_VAL, n = N_VAL;
#ifdef USE_TILED
    const char *variant = "tiled";
#else
    const char *variant = "naive";
#endif
    std::vector<float> a(m * k), b(k * n), c(m * n);
    for (int i = 0; i < m * k; ++i) a[i] = ((i % 17) - 8) * 0.125f;
    for (int i = 0; i < k * n; ++i) b[i] = ((i % 13) - 6) * 0.0625f;
    std::size_t a_bytes = a.size() * sizeof(float);
    std::size_t b_bytes = b.size() * sizeof(float);
    std::size_t c_bytes = c.size() * sizeof(float);
    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), a_bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), b_bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_c), c_bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), a_bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), b_bytes, cudaMemcpyHostToDevice));
    dim3 block(16, 16);
    dim3 grid((n + block.x - 1) / block.x,
              (m + block.y - 1) / block.y);

    // Warm-up outside the timed interval.
#ifdef USE_TILED
    matmul_tiled<<<grid, block>>>(d_a, d_b, d_c, m, k, n);
#else
    matmul_naive<<<grid, block>>>(d_a, d_b, d_c, m, k, n);
#endif
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));
#ifdef USE_TILED
    matmul_tiled<<<grid, block>>>(d_a, d_b, d_c, m, k, n);
#else
    matmul_naive<<<grid, block>>>(d_a, d_b, d_c, m, k, n);
#endif
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    float kernel_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&kernel_ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, c_bytes, cudaMemcpyDeviceToHost));

    int errors = 0;
    for (int idx = 0; idx < m * n; ++idx) {
        int row = idx / n, col = idx % n;
        double reference = 0.0;
        for (int q = 0; q < k; ++q) {
            reference += static_cast<double>(a[row * k + q]) * b[q * n + col];
        }
        if (!close_enough(c[idx], reference, 1e-4, 1e-4)) ++errors;
    }
    double gflops = (2.0 * m * k * n) / (kernel_ms * 1e6);
    std::printf("%s N=%d kernel=%.6f ms GFLOP/s=%.3f errors=%d\n",
                variant, n, kernel_ms, gflops, errors);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return errors != 0;
}



Overwriting matrix_bench.cu


In [131]:
%%bash
for n in 128 256 512; do nvcc -std=c++17 -O3 -lineinfo -arch=native -DN_VAL=$n matrix_bench.cu -o bench_naive && nvcc -std=c++17 -O3 -lineinfo -arch=native -DUSE_TILED -DN_VAL=$n matrix_bench.cu -o bench_tiled && for r in 1 2 3 4 5; do ./bench_naive; ./bench_tiled; done; done 2>&1 | tee ex7_raw.txt


naive N=128 kernel=0.032480 ms GFLOP/s=129.135 errors=0
tiled N=128 kernel=0.016288 ms GFLOP/s=257.509 errors=0
naive N=128 kernel=0.019136 ms GFLOP/s=219.184 errors=0
tiled N=128 kernel=0.015136 ms GFLOP/s=277.108 errors=0
naive N=128 kernel=0.020128 ms GFLOP/s=208.382 errors=0
tiled N=128 kernel=0.016384 ms GFLOP/s=256.000 errors=0
naive N=128 kernel=0.018816 ms GFLOP/s=222.912 errors=0
tiled N=128 kernel=0.014560 ms GFLOP/s=288.070 errors=0
naive N=128 kernel=0.020480 ms GFLOP/s=204.800 errors=0
tiled N=128 kernel=0.020480 ms GFLOP/s=204.800 errors=0
naive N=256 kernel=0.162816 ms GFLOP/s=206.088 errors=0
tiled N=256 kernel=0.069088 ms GFLOP/s=485.677 errors=0
naive N=256 kernel=0.099840 ms GFLOP/s=336.082 errors=0
tiled N=256 kernel=0.064192 ms GFLOP/s=522.720 errors=0
naive N=256 kernel=0.093440 ms GFLOP/s=359.101 errors=0
tiled N=256 kernel=0.065536 ms GFLOP/s=512.000 errors=0
naive N=256 kernel=0.099584 ms GFLOP/s=336.946 errors=0
tiled N=256 kernel=0.072160 ms GFLOP/s=465.000 e

In [132]:
import re, statistics
runs = {}
for line in open('ex7_raw.txt'):
    m = re.search(r'(naive|tiled) N=(\d+) kernel=([\d.]+) ms GFLOP/s=([\d.]+) errors=(\d+)', line)
    if m:
        runs.setdefault((m.group(1), int(m.group(2))), []).append((float(m.group(3)), float(m.group(4)), int(m.group(5))))
print(f"{'variant':>8} {'N':>5} {'kernel(ms)':>11} {'GFLOP/s':>9} {'errors':>7}")
for (v, n) in sorted(runs, key=lambda k: (k[1], k[0])):
    vals = runs[(v, n)]
    print(f"{v:>8} {n:>5} {statistics.median(x[0] for x in vals):>11.4f} {statistics.median(x[1] for x in vals):>9.2f} {max(x[2] for x in vals):>7}")


 variant     N  kernel(ms)   GFLOP/s  errors
   naive   128      0.0201    208.38       0
   tiled   128      0.0163    257.51       0
   naive   256      0.0996    336.95       0
   tiled   256      0.0655    512.00       0
   naive   512      0.8602    312.08       0
   tiled   512      0.5562    482.60       0


## Exercise 8 — Diagnostics (optional)
compute-sanitizer may be unavailable on managed GPU runtimes — the tutorial says to record that limitation.


In [133]:
!which compute-sanitizer >/dev/null 2>&1 && for p in "./vector_add" "./reduce_sum" "./matrix_mul_tiled"; do echo "== memcheck $p"; compute-sanitizer --tool memcheck $p 2>&1 | tail -3; done || echo "compute-sanitizer not available on this runtime (recorded limitation)"


== memcheck ./vector_add
n=1000003 blocks=3907 threads=256 errors=0
first=0.00 last=9.25
========= ERROR SUMMARY: 0 errors
== memcheck ./reduce_sum
========= COMPUTE-SANITIZER
n=1000003 expected=3000003 result=3000003 match=yes
========= ERROR SUMMARY: 0 errors
== memcheck ./matrix_mul_tiled
========= COMPUTE-SANITIZER
TILED M=127 K=130 N=129 errors=0 max_error=0.000e+00
========= ERROR SUMMARY: 0 errors


### Controlled bug experiments (optional)
**Bug 1:** remove the `i < n` guard in a *copy*, run memcheck with n=257, then restore.


In [134]:
!cp vector_add.cu vector_add_buggy.cu
!python3 -c "s=open('vector_add_buggy.cu').read().replace('if (i < n) c[i] = a[i] + b[i];','c[i] = a[i] + b[i];',1);open('vector_add_buggy.cu','w').write(s)"
!grep -n 'c[i] = a[i]' vector_add_buggy.cu | head -2



In [135]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native -DN_VAL=257 vector_add_buggy.cu -o vector_add_buggy && compute-sanitizer --tool memcheck ./vector_add_buggy 2>&1 | tail -5; echo "--- restoring ---"


=========         Host Frame: main [0x8dbb] in vector_add_buggy
vector_add_buggy.cu:36: CUDA error: unspecified launch failure
========= Target application returned an error
========= ERROR SUMMARY: 97 errors
--- restoring ---


**Bug 2:** remove the first `__syncthreads()` in a reduction *copy*, run racecheck, then restore.


In [136]:
!cp reduce_sum.cu reduce_sum_buggy.cu
!sed -i '0,/^    __syncthreads();$/s///' reduce_sum_buggy.cu
!nvcc -std=c++17 -O3 -lineinfo -arch=native reduce_sum_buggy.cu -o reduce_sum_buggy && compute-sanitizer --tool racecheck ./reduce_sum_buggy 2>&1 | tail -5; echo "--- originals untouched; rerun the Exercise 3/6 cells to confirm ---"


=========     and Write access at block_sum(const int *, int *, int)+0x100 in reduce_sum_buggy.cu:16 [2000384 hazards]
n=1000003 expected=3000003 result=2648818 match=no
========= Target application returned an error
========= RACECHECK SUMMARY: 1 hazard displayed (1 error, 0 warnings)
--- originals untouched; rerun the Exercise 3/6 cells to confirm ---


## Mini-project — 2D five-point stencil


In [137]:
%%writefile stencil.cu
#include "cuda_check.cuh"
#include <vector>
#include <chrono>

// Mini-project: 2D five-point stencil (one simulation step / image filter).
// y[r][c] = (x[r][c] + x[r-1][c] + x[r+1][c] + x[r][c-1] + x[r][c+1]) / 5
// Boundary cells are copied unchanged. Separate input/output arrays.
// Usage: ./stencil [H W steps]   (defaults: 257 259 10)

__global__ void stencil_step(const float *in, float *out, int h, int w)
{
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    int r = blockIdx.y * blockDim.y + threadIdx.y;
    if (r >= h || c >= w) return;
    int i = r * w + c;
    if (r == 0 || r == h - 1 || c == 0 || c == w - 1) {
        out[i] = in[i];
        return;
    }
    out[i] = (in[i] + in[i - w] + in[i + w] + in[i - 1] + in[i + 1]) * 0.2f;
}

static void stencil_cpu(const float *in, float *out, int h, int w)
{
    for (int r = 0; r < h; ++r) {
        for (int c = 0; c < w; ++c) {
            int i = r * w + c;
            if (r == 0 || r == h - 1 || c == 0 || c == w - 1) {
                out[i] = in[i];
            } else {
                out[i] = (in[i] + in[i - w] + in[i + w] +
                          in[i - 1] + in[i + 1]) * 0.2f;
            }
        }
    }
}

int main(int argc, char **argv)
{
    const int h = (argc > 1) ? std::atoi(argv[1]) : 257;
    const int w = (argc > 2) ? std::atoi(argv[2]) : 259;
    const int steps = (argc > 3) ? std::atoi(argv[3]) : 10;
    const std::size_t bytes = static_cast<std::size_t>(h) * w * sizeof(float);

    std::vector<float> in(h * w), ref(h * w), ref2(h * w), out(h * w);
    for (int i = 0; i < h * w; ++i)
        in[i] = ((i % 41) - 20) * 0.25f;

    // CPU reference: same number of steps, ping-pong on host.
    std::vector<float> ping = in, pong(h * w);
    auto cpu_start = std::chrono::steady_clock::now();
    for (int s = 0; s < steps; ++s) {
        stencil_cpu(ping.data(), pong.data(), h, w);
        ping.swap(pong);
    }
    double cpu_ms = std::chrono::duration<double, std::milli>(
        std::chrono::steady_clock::now() - cpu_start).count();
    ref = ping;

    float *d_a = nullptr, *d_b = nullptr;
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_a), bytes));
    CUDA_CHECK(cudaMalloc(reinterpret_cast<void **>(&d_b), bytes));
    dim3 block(16, 16);
    dim3 grid((w + block.x - 1) / block.x,
              (h + block.y - 1) / block.y);

    // Variant A: data stays on GPU; swap buffers between steps.
    CUDA_CHECK(cudaMemcpy(d_a, in.data(), bytes, cudaMemcpyHostToDevice));
    stencil_step<<<grid, block>>>(d_a, d_b, h, w); // warm-up
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    auto wall_a = std::chrono::steady_clock::now();
    CUDA_CHECK(cudaMemcpy(d_a, in.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(start));
    float *src = d_a, *dst = d_b;
    for (int s = 0; s < steps; ++s) {
        stencil_step<<<grid, block>>>(src, dst, h, w);
        CUDA_CHECK(cudaGetLastError());
        float *tmp = src; src = dst; dst = tmp;
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaMemcpy(out.data(), src, bytes, cudaMemcpyDeviceToHost));
    double opA_ms = std::chrono::duration<double, std::milli>(
        std::chrono::steady_clock::now() - wall_a).count();
    float kernA_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&kernA_ms, start, stop));

    int errors = 0;
    for (int i = 0; i < h * w; ++i)
        if (!close_enough(out[i], ref[i], 1e-4, 1e-4)) ++errors;

    // Variant B: H2D + kernel + D2H repeated for every step.
    auto wall_b = std::chrono::steady_clock::now();
    src = d_a; dst = d_b;
    for (int s = 0; s < steps; ++s) {
        CUDA_CHECK(cudaMemcpy(src, (s == 0 ? in.data() : out.data()),
                              bytes, cudaMemcpyHostToDevice));
        stencil_step<<<grid, block>>>(src, dst, h, w);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());
        CUDA_CHECK(cudaMemcpy(out.data(), dst, bytes, cudaMemcpyDeviceToHost));
        float *tmp = src; src = dst; dst = tmp;
    }
    double opB_ms = std::chrono::duration<double, std::milli>(
        std::chrono::steady_clock::now() - wall_b).count();
    int errors_b = 0;
    for (int i = 0; i < h * w; ++i)
        if (!close_enough(out[i], ref[i], 1e-4, 1e-4)) ++errors_b;

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));

    std::printf("stencil H=%d W=%d steps=%d\n", h, w, steps);
    std::printf("CPU %d steps: %.3f ms\n", steps, cpu_ms);
    std::printf("GPU resident : kernel-all-steps=%.3f ms transfer-inclusive=%.3f ms errors=%d\n",
                kernA_ms, opA_ms, errors);
    std::printf("GPU per-step : transfer-inclusive=%.3f ms errors=%d\n", opB_ms, errors_b);
    return (errors != 0 || errors_b != 0);
}



Overwriting stencil.cu


In [138]:
!nvcc -std=c++17 -O3 -lineinfo -arch=native stencil.cu -o stencil
!./stencil 257 259 10


stencil H=257 W=259 steps=10
CPU 10 steps: 1.542 ms
GPU resident : kernel-all-steps=0.067 ms transfer-inclusive=0.253 ms errors=0
GPU per-step : transfer-inclusive=1.769 ms errors=0


### Larger sizes


In [139]:
!./stencil 1024 1024 20


stencil H=1024 W=1024 steps=20
CPU 20 steps: 49.729 ms
GPU resident : kernel-all-steps=0.925 ms transfer-inclusive=3.035 ms errors=0
GPU per-step : transfer-inclusive=36.338 ms errors=0


## Done — what to send back
**File → Download as .ipynb** and send me the downloaded file. It contains every program *and* all real outputs (device name, timings, validation results) — that's the evidence I need to compute the final medians, draw the plots, and write the report.

If any cell fails, send the notebook anyway plus a note about which cell failed.
